# SilverWork_incremental_industrial_v2
Incremental Silver processing using only new Bronze rows.


## Step 1 — Imports and setup
This cell imports Spark, Window, and Delta helpers, switches to the right catalog, makes sure the **Silver schema** exists, and creates a `silver_run_id` for the current run.


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid
from pyspark.sql import Window
# Import shared Spark transformations from this deployed project folder.
import sys
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
workspace_parent = notebook_path.rsplit("/", 1)[0]
if not workspace_parent.startswith("/Workspace/"):
    workspace_parent = "/Workspace" + workspace_parent
if workspace_parent not in sys.path:
    sys.path.insert(0, workspace_parent)
from novacart_transformations import clean_order_batch, clean_product_batch, clean_payment_batch


In [0]:
spark.sql("use catalog adb_datatocrunchproject")
spark.sql("create schema if not exists silver_schema")
silver_run_id = str(uuid.uuid4())
print("Current Silver Run ID: ", silver_run_id)

## Step 2 — Silver control table
This table stores the Silver processing history: one row is appended per entity per run. The latest SUCCESS row (by `updated_at`) is the current watermark.

It helps us track:
* the latest Bronze run already processed by Silver
* the latest Bronze ingestion timestamp already processed
* how many rows were merged in each Silver run


In [0]:
spark.sql("""
    create table if not exists adb_datatocrunchproject.silver_schema.processing_control(
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,
        silver_run_id string,
        updated_at timestamp
    )
    using delta
""")


## Step 3 — Helper functions
This cell contains reusable logic for Silver:

* `upsert_to_silver()` merges cleaned / transformed rows into the Silver target table
* `get_last_processed_bronze_ingested_at()` reads the Silver watermark
* `upsert_silver_control()` updates the Silver control table
* `get_incremental_bronze()` reads only new Bronze rows that Silver has not processed yet


In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
            .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        df_source.write.saveAsTable(target_table)

def upsert_quarantine(df_source, target_table):
    """Persist quarantine records once per source key/version, including on retries."""
    if spark.catalog.tableExists(target_table):
        target_columns = set(spark.table(target_table).columns)
        if "quarantine_id" not in target_columns:
            raise ValueError(
                f"{target_table} predates deterministic quarantine IDs; "
                "recreate or migrate this quarantine table before retry-safe processing."
            )
        dt = DeltaTable.forName(spark, target_table)
        (
            dt.alias("target")
            .merge(df_source.alias("source"), "target.quarantine_id = source.quarantine_id")
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df_source.write.format("delta").mode("errorifexists").saveAsTable(target_table)


In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl = (
        spark.table("adb_datatocrunchproject.silver_schema.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        return None, None
    return (
        rows[0]["last_processed_bronze_ingested_at"],
        rows[0]["last_processed_bronze_run_id"]
    )

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "SUCCESS",
                silver_run_id,
                datetime.utcnow(),
            )
        ],
        schema="""
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at timestamp
        """
    )

    # Append one row per run so the control table keeps a full run history.
    # get_last_processed_bronze_ingested_at() reads the latest SUCCESS row by updated_at.
    ctrl_df.write.format("delta").mode("append").saveAsTable(
        "adb_datatocrunchproject.silver_schema.processing_control"
    )


In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at, last_run_id = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at, last_run_id

    return (
        bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)),
        last_ingested_at,
        last_run_id
    )


**Step 4 — Orders incremental processing**

This cell processes **orders** from Bronze to Silver.

It does the following:
* reads only new Bronze order rows
* cleans values like `order_status` and `order_amount`
* keeps only the latest version per `order_id`
* validates business rules

** • sends bad rows to quarantine
• merges good rows into orders_transformed


In [0]:
# Step 4 - Orders incremental processing
# Read only the Bronze order rows that Silver has not processed yet.
orders_inc, last_orders_ingested_at, last_order_run_id = get_incremental_bronze(
    "adb_datatocrunchproject.bronze_schema.orders_raw",
    "orders"
)

# Count the incremental order rows entering Silver in this run.
orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver = {orders_inc_count}")

# Only run Silver order cleaning and validation when there are new Bronze order rows.
if orders_inc_count > 0:
    # Shared, unit-tested transformation: clean, deduplicate, validate, and split.
    orders_cleaned, orders_good, orders_bad = clean_order_batch(orders_inc, silver_run_id)

    # Persist the cleaned and deduplicated orders (before validation) as their own Silver table.
    upsert_to_silver(
        orders_cleaned,
        "adb_datatocrunchproject.silver_schema.orders_cleaned",
        "order_id"
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_good,
        "adb_datatocrunchproject.silver_schema.orders_transformed",
        "order_id"
    )

    # Merge quarantined order versions by deterministic source-version ID; retries do not append duplicates.
    upsert_quarantine(
        orders_bad,
        "adb_datatocrunchproject.silver_schema.orders_quarantine"
    )

    mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        orders_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())

else:
    print("No new orders Bronze rows for Silver.")
    
    upsert_silver_control(
        "orders",
        last_order_run_id,
        last_orders_ingested_at,
        orders_inc_count
    )


**
Step 5 — Products incremental processing
This cell processes products from Bronze to Silver.
It handles:
• product name cleanup
• category standardization
• price cleanup and numeric conversion
• latest-record selection per product_id
• data quality validation
• quarantine for bad rows
• merge into Silver current-state tables

In [0]:
# Step 5 - Products incremental processing
# Read only the Bronze product rows that Silver has not processed yet.
products_inc, last_products_ingested_at, last_products_run_id = get_incremental_bronze("adb_datatocrunchproject.bronze_schema.products_raw", "products")

# Count the incremental product rows entering Silver in this run.
products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver = {products_inc_count}")

if products_inc_count > 0:
    # Shared, unit-tested transformation: clean, deduplicate, validate, and split.
    products_cleaned, products_good, products_bad = clean_product_batch(products_inc, silver_run_id)
    upsert_to_silver(
        products_cleaned,
        "adb_datatocrunchproject.silver_schema.products_cleaned",
        "product_id"
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(products_good, "adb_datatocrunchproject.silver_schema.products_transformed", "product_id")
    
    # Merge quarantined product versions by deterministic source-version ID; retries do not append duplicates.
    upsert_quarantine(products_bad, "adb_datatocrunchproject.silver_schema.products_quarantine")

    mx_ingested = products_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = products_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control("products", mx_run, mx_ingested, products_good.count())
else:
    print("No new products Bronze rows for Silver.")
    upsert_silver_control(
        "products",
        last_products_run_id,
        last_products_ingested_at,
        products_inc_count
    )


## Step 6 — Payments incremental processing
This cell processes payments from Bronze to Silver.
It cleans:
• payment_status
• paid_amount
• processed_at
Then it validates records, quarantines bad rows, and merges valid rows into the Silver transformed payments table.

In [0]:
# Step 6 - Payments incremental processing
# Read only the Bronze payment rows that Silver has not processed yet.
payments_inc, last_payments_ingested_at, last_payment_run_id = get_incremental_bronze("adb_datatocrunchproject.bronze_schema.payments_raw", "payments")
print("Payments last processed Bronze ingested_at = ", last_payments_ingested_at)
#print("Payments incremental Bronze rows =", payments_inc.count())

# Count the incremental payment rows entering Silver in this run.
payments_inc_count = payments_inc.count()
print(f"payments rows_to_process_in_silver = {payments_inc_count}")

if payments_inc.count() > 0:
    # Shared, unit-tested transformation: clean, deduplicate, validate, and split.
    payments_cleaned, payments_good, payments_bad = clean_payment_batch(payments_inc, silver_run_id)
    upsert_to_silver(
        payments_cleaned,
        "adb_datatocrunchproject.silver_schema.payments_cleaned",
        "payment_id"
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(payments_good, "adb_datatocrunchproject.silver_schema.payments_transformed", "payment_id")

    # Merge quarantined payment versions by deterministic source-version ID; retries do not append duplicates.
    upsert_quarantine(payments_bad, "adb_datatocrunchproject.silver_schema.payments_quarantine")

    mx_ingested = payments_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = payments_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control("payments", mx_run, mx_ingested, payments_good.count())
else:
    print("No new payments Bronze rows for Silver.")
    upsert_silver_control(
        "payments",
        last_payment_run_id,
        last_payments_ingested_at,
        payments_inc_count
    )


Step 7 — Quick validation
This final cell prints Silver transformed row counts and shows the Silver control table so you can confirm the incremental processing behavior.


In [0]:
print("Products transformed count :", spark.sql("SELECT COUNT(*) FROM adb_datatocrunchproject.silver_schema.products_transformed").collect()[0][0])
print("Orders transformed count :", spark.sql("SELECT COUNT(*) FROM adb_datatocrunchproject.silver_schema.orders_transformed").collect()[0][0])
print("Payments transformed count :", spark.sql("SELECT COUNT(*) FROM adb_datatocrunchproject.silver_schema.payments_transformed").collect()[0][0])
display(
    spark.table("adb_datatocrunchproject.silver_schema.processing_control")
    .orderBy("entity_name", F.col("updated_at").desc())
)
